# Study report: timeout-queue-cost

The study measures how a load changes the IRQ latency (`irq`, M1) and
the ISR to thread wake latency (`isr2thr`, M2). It asks which load makes
the wake slower, and how much each periodic timer and each load thread
adds. It also asks how the timeout queue backend changes the latency
with 256 periodic timers. All experiments use the app `wake_latency`, with a semaphore
given from the wrapped ISR, and the board defaults: `SCHED_SIMPLE`, time
slicing on, and the timeout queue backend DLIST.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. LATBENCH_INSTANCE: the instance folder ("study.py analysis notebooks" sets it). Else
# LATBENCH_RESULTS: the results root of a lab space (for example <lab>/results/kernel-latency), and the
# notebook finds its instance in it. LATBENCH_REPO: the safety repo; else the repo of the current folder.
import os
import subprocess
import sys
from pathlib import Path

INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or Path(
    os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-load-scaling@nucleo_g474re/st-timeout-queue-cost")
REPO = Path(os.environ.get("LATBENCH_REPO") or subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## The loads and the mechanisms

The five kinds of load (`study.md`):

- Blocked threads: each load thread waits forever on a semaphore of its
  own (priority 5).
- Sleeper threads: each load thread sleeps 1..20 ms in a loop (priority
  5). Each sleep adds a timeout and a thread switch.
- Busy threads: each load thread runs a busy loop at the lowest
  priority. With time slicing, the busy threads take turns at each slice
  expiry.
- Periodic timers: N periodic `k_timer`s, period 10..100 ms, random
  start. Their expiry functions run in the SysTick ISR. Each restart
  inserts a timeout into the timeout queue.
- Far timers: 256 one-shot timers one hour away. They change only the
  length of the timeout queue.

The mechanisms:

1. The DLIST backend keeps the timeouts in the order of expiry. An
   insert walks only the timeouts that expire before the new one.
2. With time slicing, each wake resets the time slice, and the reset
   inserts a timeout. Without time slicing, the wake has no slice reset.
3. With `SCHED_SIMPLE`, the woken thread has the highest priority. Thus
   its insert into the ready queue does not walk the busy threads.

## H1 a, b, c: load threads (wake-threads-1m)

The study `zephyr-versions` owns `wake-threads-1m`. The points are the
cube `kind` x `threads` and the point `none-n0`. `none-n0` has no load
threads: it is the reference, in the same binary as the other points,
apart from the load. Its axis value `n0` has no number, so the figure
shows it as horizontal lines.

- H1 a (manual): blocked load threads cost nothing.
- H1 b (manual): with `SCHED_SIMPLE` the woken thread has the highest
  priority, so its insert into the ready queue is O(1). The busy points
  show this.
- H1 c: the `isr2thr` mean increases from `none-n0` over the sleeper
  points.

The central result `thread-slope` is the least-squares slope of the p99
over the number of threads, for each kind. The dashed lines in the
second figure are this fit. The tables give the fit and all points.

In [ ]:
import numpy as np
import plotly.graph_objects as go


def ref_lines(fig, ref, stats):
    """Horizontal lines at the statistics of the point none-n0 (its axis value has no number)."""
    for st, dash in zip(stats, ("dash", "dot")):
        if len(ref):
            fig.add_hline(y=ref[st].iloc[0], line_dash=dash, line_color="#6e6d66",
                          annotation_text=f"none-n0 {st}", annotation_position="top left")
    return fig


if nbfig.built(inst, "wake-threads-1m"):
    th = inst.rows("wake-threads-1m")
    m2 = th[th.metric == "isr2thr"]
    loaded = m2[m2["value.threads"].notna()]
    ref = m2[m2.point == "none-n0"]
    ref_lines(nbfig.sweep(loaded, "value.threads", "isr2thr mean over the load threads (H1 c)", stats=("mean",),
                          group="axis.kind"), ref, ("mean",)).show()

In [ ]:
if nbfig.built(inst, "wake-threads-1m"):
    fig = nbfig.sweep(loaded, "value.threads", "isr2thr p99 over the load threads, with the fit (thread-slope)",
                      stats=("p99",), group="axis.kind")
    fits = []
    for k, (kind, sub) in enumerate(loaded.groupby("axis.kind", sort=False)):
        slope, icept = np.polyfit(sub["value.threads"], sub["p99"], 1)
        fits.append({"kind": kind, "p99 slope (cycles per thread)": round(slope, 1), "intercept": round(icept, 1)})
        xs = [sub["value.threads"].min(), sub["value.threads"].max()]
        fig.add_trace(go.Scatter(x=xs, y=[slope * x + icept for x in xs], mode="lines", name=f"{kind} fit",
                                 line={"color": nbfig.color(k), "width": 1, "dash": "dash"}, hoverinfo="skip"))
    ref_lines(fig, ref, ("p99",)).show()
    display(pd.DataFrame(fits).set_index("kind"))
    display(nbfig.pivot(th, ["p50", "p99", "mean", "max"]))

## H1 d and H2 a: load timers (wake-timers-1m)

The study `zephyr-versions` owns `wake-timers-1m`. The points have 0, 16,
64 and 256 periodic load timers, with time slicing (`default`: each wake
resets the time slice) and without (`noslice`). `t256far` has 256
one-shot timers one hour away.

- H1 d (manual): an insert into the DLIST timeout queue walks only the
  timeouts that expire before the new one. Thus `t256far` is as `t0`,
  with and without time slicing.
- H2 a: periodic timers cost in the timer ISR (restart) and in the slice
  reset of each wake. Thus with time slicing the `isr2thr` mean
  increases with the number of timers.

The central result `timer-slope` is the least-squares slope of the
`isr2thr` p99 over the number of timers, with time slicing, without
`t256far`. The figure shows `t256far` as single markers at its number of
timers. The menu selects the metric.

In [ ]:
if nbfig.built(inst, "wake-timers-1m"):
    tm = inst.rows("wake-timers-1m")
    near = tm[~tm.point.str.startswith("t256far")]
    fig = nbfig.sweep(near, "value.timers", "ISR to thread over the load timers (wake-timers-1m)",
                      stats=("mean", "p99"), group="axis.anchor")
    far = tm[tm.point.str.startswith("t256far")]
    metrics = nbfig.metric_order(near.metric)
    per = len(fig.data) // len(metrics)
    for i, m in enumerate(metrics):
        for j, st in enumerate(("mean", "p99")):
            sub = far[far.metric == m]
            fig.add_trace(go.Scatter(x=sub["value.timers"], y=sub[st], mode="markers", name=f"t256far {st}",
                                     visible=(i == 0),
                                     customdata=sub.point, marker={"size": 11, "symbol": ["x", "cross"][j],
                                                                   "color": "#6e6d66"},
                                     hovertemplate="%{customdata}: %{y} cycles<extra>t256far " + st + "</extra>"))
    # the metric menu: keep the t256far markers of the selected metric visible
    n = len(fig.data)
    for i, b in enumerate(fig.layout.updatemenus[0].buttons if fig.layout.updatemenus else []):
        b.args[0]["visible"] = [j // per == i for j in range(len(metrics) * per)] + \
                               [(j - len(metrics) * per) // 2 == i for j in range(len(metrics) * per, n)]
    fig.show()
    display(nbfig.pivot(tm, ["p50", "p99", "mean", "max"]))

## The timeout queue backends with 256 periodic timers

`wake-timeout-backends-1m` (owned by `zephyr-versions`) measures the
backends DLIST, MINHEAP, WHEEL and BUCKET. Its point `dlist` is a
reference in the same binary as the other points.
`wake-timeout-skiplist-1m` (owned by this study) measures SKIPLIST: the
SKIPLIST backend is not in all Zephyr versions. The point `t256-default`
of `wake-timers-1m` has the same load with the DLIST backend. These
experiments stated no prediction. Their tail is the central result
`backend-tail`. The last table gives the run and the start of
`image_sha256` of each point.

In [ ]:
parts = []
for exp in ("wake-timeout-backends-1m", "wake-timeout-skiplist-1m", "wake-timers-1m"):
    if nbfig.built(inst, exp):
        d = inst.rows(exp).assign(experiment=exp)
        if exp == "wake-timers-1m":
            d = d[d.point == "t256-default"]
        parts.append(d)
bk = pd.concat(parts)
bk["point"] = bk["point"] + " (" + bk["experiment"] + ")"
nbfig.dots(bk, "irq and isr2thr per timeout queue backend, 256 periodic timers",
           stats=("p50", "p99", "p99.99", "max")).show()
display(nbfig.pivot(bk, ["p50", "p99", "p99.99", "max", "mean"]))
bk.drop_duplicates("point").assign(image=lambda d: d.image_sha256.str[:12]).set_index("point")[["run", "image"]]

In [ ]:
series = {}
for exp in ("wake-timeout-backends-1m", "wake-timeout-skiplist-1m"):
    if nbfig.built(inst, exp):
        series.update(nbfig.hists_of(inst, exp))
if nbfig.built(inst, "wake-timers-1m"):
    series["t256-default (wake-timers-1m)"] = next(rep["hists"] for _, _, p, rep in inst.points("wake-timers-1m")
                                                   if p == "t256-default")
nbfig.ccdf(series, "Timeout queue backends: CCDF", ["isr2thr", "irq", "irq2thr"],
           dashed={"t256-default (wake-timers-1m)"}).show()